# 02 — Regional Expression Profiles

Detailed expression patterns of mechanosensitive ion channels across brain regions,
averaged across all 6 Allen Brain Atlas microarray donors.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from src import gene_lists, data_loader

sns.set_theme(style='whitegrid', context='notebook')
%matplotlib inline

## 1. Load expression averaged across donors

We aggregate the microarray expression across all 6 donors, averaging samples within each brain region.

In [ ]:
genes = list(gene_lists.ALL_MECHANOSENSITIVE.keys())
region_expr = data_loader.get_microarray_region_means(genes)
print(f"Expression matrix: {region_expr.shape[0]} genes x {region_expr.shape[1]} regions")
region_expr.iloc[:5, :5]

In [ ]:
# Drop regions with too few data points (NaN-heavy columns)
threshold = 0.5  # keep regions present in at least 50% of genes
valid_cols = region_expr.columns[region_expr.notna().mean() >= threshold]
region_expr_clean = region_expr[valid_cols]
print(f"After filtering: {region_expr_clean.shape[1]} regions retained")

## 2. Top regions for FUS-core genes

Focus on genes most directly implicated in FUS mechanotransduction.

In [ ]:
core_genes = list(gene_lists.FUS_CORE_GENES.keys())
core_expr = region_expr_clean.loc[region_expr_clean.index.isin(core_genes)]

# Z-score normalize per gene
from src.clustering import zscore_expression
core_z = zscore_expression(core_expr)

# Display names
core_z.index = [gene_lists.get_display_name(g) for g in core_z.index]

fig, ax = plt.subplots(figsize=(20, 6))
sns.heatmap(
    core_z,
    cmap='RdBu_r', center=0,
    xticklabels=True, yticklabels=True,
    ax=ax, linewidths=0.2,
    cbar_kws={'label': 'Z-score', 'shrink': 0.6}
)
ax.set_title('FUS Core Mechanosensitive Channels — Regional Expression (Z-scored)', fontsize=13)
ax.tick_params(axis='x', rotation=90, labelsize=6)
ax.tick_params(axis='y', labelsize=10)
plt.tight_layout()

## 3. Top expressing regions for each core gene

In [ ]:
n_top = 10
for gene in core_genes:
    if gene in region_expr_clean.index:
        row = region_expr_clean.loc[gene].dropna().sort_values(ascending=False)
        display_name = gene_lists.get_display_name(gene)
        print(f"\n--- {display_name} — Top {n_top} regions ---")
        for region, val in row.head(n_top).items():
            print(f"  {region:>15s}: {val:.2f}")

## 4. Piezo expression across brain regions

In [ ]:
piezo_genes = list(gene_lists.PIEZO_GENES.keys())
piezo_expr = region_expr_clean.loc[region_expr_clean.index.isin(piezo_genes)]

fig, axes = plt.subplots(len(piezo_genes), 1, figsize=(16, 4 * len(piezo_genes)))
if len(piezo_genes) == 1:
    axes = [axes]

for ax, gene in zip(axes, piezo_genes):
    if gene not in piezo_expr.index:
        continue
    row = piezo_expr.loc[gene].dropna().sort_values(ascending=False)
    top_n = row.head(30)
    ax.bar(range(len(top_n)), top_n.values, color='#E74C3C', alpha=0.8)
    ax.set_xticks(range(len(top_n)))
    ax.set_xticklabels(top_n.index, rotation=90, fontsize=8)
    ax.set_ylabel('Expression')
    ax.set_title(f'{gene_lists.get_display_name(gene)} — Top 30 Regions', fontsize=12)

plt.tight_layout()

## 5. K2P mechanosensitive channels (TREK-1, TREK-2, TRAAK)

In [ ]:
k2p_mechano = list(gene_lists.K2P_MECHANOSENSITIVE.keys())
k2p_expr = region_expr_clean.loc[region_expr_clean.index.isin(k2p_mechano)]

fig, axes = plt.subplots(len(k2p_mechano), 1, figsize=(16, 4 * len(k2p_mechano)))
for ax, gene in zip(axes, k2p_mechano):
    if gene not in k2p_expr.index:
        continue
    row = k2p_expr.loc[gene].dropna().sort_values(ascending=False)
    top_n = row.head(30)
    ax.bar(range(len(top_n)), top_n.values, color='#3498DB', alpha=0.8)
    ax.set_xticks(range(len(top_n)))
    ax.set_xticklabels(top_n.index, rotation=90, fontsize=8)
    ax.set_ylabel('Expression')
    ax.set_title(f'{gene_lists.get_display_name(gene)} — Top 30 Regions', fontsize=12)

plt.tight_layout()

## 6. Gene–gene correlation across brain regions

In [ ]:
# Correlation of all mechanosensitive genes with each other across brain regions
gene_corr = region_expr_clean.T.corr(method='spearman')
display_index = [gene_lists.get_display_name(g) for g in gene_corr.index]
gene_corr.index = display_index
gene_corr.columns = display_index

fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(
    gene_corr, cmap='RdBu_r', center=0,
    square=True, linewidths=0.3,
    ax=ax,
    cbar_kws={'label': 'Spearman ρ', 'shrink': 0.6}
)
ax.set_title('Spatial Correlation Between Mechanosensitive Channels', fontsize=14)
ax.tick_params(axis='x', rotation=90, labelsize=8)
ax.tick_params(axis='y', labelsize=8)
plt.tight_layout()